# Comparación con NLP: spaCy y Transformers

**Responsable:** Persona 1

Se procesa el conjunto de prueba (`data/test_set.csv`) con dos herramientas de NLP para español y se
guardan las salidas en el formato acordado (`docs/gramatica_es.md`, sección 2.4):

| Herramienta | Salida |
| --- | --- |
| spaCy `es_core_news_sm` | `results/spacy.jsonl` |
| Modelo de HuggingFace Transformers para español | `results/transformers.jsonl` |

spaCy no decide si la oración es válida (`acepta` = `null`): se compara por **etiquetado** y por
**funciones sintácticas** (sujeto, verbo y objeto). Transformers etiqueta con un modelo y **juzga la
gramaticalidad** con otro (BETO, sección 2.5), así que sí llena `acepta`. La conversión a nuestras categorías es la de
la sección 2.5, implementada en `experiments/tagsets.py`.

**Antes de correrlo** (una sola vez, con el `.venv` del proyecto activado):

```bash
pip install -r requirements.txt -r requirements-experiments.txt
python -m spacy download es_core_news_sm
```

Preguntas que este notebook debe responder (van al informe):

1. ¿En qué oraciones spaCy y Transformers analizan bien lo que la CFG **no puede**? Pista: `en_gramatica = no` y `espanol_valido = si`.
2. ¿En qué oraciones la CFG acierta y ellos **no**?

In [ ]:
# Configuración: el notebook vive en notebooks/, el código del proyecto en la raíz.
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from evaluation.formats import Prediction, load_predictions, load_test_set, results_path, save_predictions

pd.set_option("display.max_colwidth", 80)
FIGURES = ROOT / "results" / "figures"     # figuras para el informe
REPORTS = ROOT / "results" / "reports"     # tablas para el informe
REPORTS.mkdir(parents=True, exist_ok=True)
items = load_test_set()
by_id = {item.id: item for item in items}
print(f"{len(items)} oraciones en el conjunto de prueba")

In [ ]:
pd.DataFrame([{
    "id": i.id, "nivel": i.level, "oracion": i.sentence,
    "en_gramatica": i.in_grammar, "espanol_valido": i.valid_spanish,
    "tipo_error": i.error_type, "fenomeno": ";".join(i.phenomena),
} for i in items])

## 1. spaCy

### 1.1 Correr spaCy y guardar `results/spacy.jsonl`

In [ ]:
import spacy

try:
    nlp = spacy.load("es_core_news_sm")
except OSError:
    raise SystemExit("Falta el modelo: python -m spacy download es_core_news_sm")

# Anotar en el informe: versión de spaCy y del modelo.
print("spaCy", spacy.__version__, "| modelo", nlp.meta["lang"] + "_" + nlp.meta["name"], nlp.meta["version"])

In [ ]:
from experiments.tagsets import spacy_output

docs = {}  # id -> Doc de spaCy, para el análisis de abajo
spacy_predictions = []
for item in items:
    start = time.perf_counter()
    try:
        doc = nlp(item.sentence)
        docs[item.id] = doc
        row = spacy_output(item.id, doc, round((time.perf_counter() - start) * 1000, 3))
        spacy_predictions.append(Prediction.from_json(row))
    except Exception as e:  # el formato pide escribir la línea aunque falle
        spacy_predictions.append(Prediction(item.id, "spacy", accepts=None,
                                            run_error=f"{type(e).__name__}: {e}"))

print("Guardado en", save_predictions("spacy", spacy_predictions, items))

### 1.2 Etiquetas y funciones sintácticas, oración por oración

Compara la salida de spaCy (ya convertida a nuestras categorías) con lo esperado en el conjunto de prueba.
Las métricas oficiales las calcula la Persona 4 con `evaluation/metrics.py`; esta tabla es para leer los
casos uno por uno.

In [ ]:
rows = []
for item, pred in zip(items, spacy_predictions):
    expected = [c for _, c in item.tags]
    got = [c for _, c in (pred.tags or [])]
    wrong = [f"{w}: {e}→{g}" for (w, e), g in zip(item.tags, got) if e != g] if len(got) == len(expected) else ["distinto número de palabras"]
    rows.append({
        "id": item.id,
        "oracion": item.sentence,
        "en_gramatica": item.in_grammar,
        "etiquetas_ok": not wrong,
        "errores_etiqueta": ", ".join(wrong),
        "sujeto": f"{pred.subject!r} (esperado {item.subject!r})",
        "verbo": f"{pred.verb!r} (esperado {item.verb!r})",
        "objeto": f"{pred.object!r} (esperado {item.object!r})",
        "funciones_ok": (pred.subject, pred.verb, pred.object) == (item.subject, item.verb, item.object),
    })
table = pd.DataFrame(rows)
table

In [ ]:
total_words = sum(len(i.tags) for i in items)
right_words = sum(
    sum(e == g for (_, e), (_, g) in zip(i.tags, p.tags))
    for i, p in zip(items, spacy_predictions) if p.tags and len(p.tags) == len(i.tags)
)
print(f"Etiquetas correctas: {right_words}/{total_words} palabras")
print(f"Oraciones con todas las etiquetas correctas: {table.etiquetas_ok.sum()}/{len(table)}")
print(f"Oraciones con sujeto, verbo y objeto correctos: {table.funciones_ok.sum()}/{len(table)}")
for in_grammar, group in table.groupby("en_gramatica"):
    print(f"  en_gramatica={in_grammar}: etiquetas OK {group.etiquetas_ok.sum()}/{len(group)}, "
          f"funciones OK {group.funciones_ok.sum()}/{len(group)}")

### 1.3 Análisis completo de spaCy

Lo que spaCy ve antes de convertirlo a nuestras categorías: etiqueta UPOS, relación de dependencia, núcleo y
rasgos morfológicos. Sirve para explicar los errores de la tabla anterior.

In [ ]:
def spacy_detail(doc) -> pd.DataFrame:
    return pd.DataFrame([{
        "palabra": t.text, "upos": t.pos_, "dep": t.dep_, "nucleo": t.head.text,
        "genero": ",".join(t.morph.get("Gender")), "numero": ",".join(t.morph.get("Number")),
        "rasgos": str(t.morph),
    } for t in doc if t.pos_ != "PUNCT"])


for item in items:
    print(f"\n{item.id}. {item.sentence}   (en_gramatica={item.in_grammar})")
    print(spacy_detail(docs[item.id]).to_string(index=False))

### 1.4 Experimento: ¿podría spaCy detectar los errores de concordancia?

spaCy no rechaza oraciones, pero sí da rasgos de género y número. Aquí se prueba si con esos rasgos se
podrían detectar los mismos errores que detecta nuestro analizador semántico, con las reglas de la
sección 1.3:

- `det` y `amod` deben concordar en género y número con su núcleo;
- el sujeto (`nsubj`) debe concordar en número con el verbo raíz.

Es un experimento de análisis: **no cambia `results/spacy.jsonl`**, donde `acepta` sigue siendo `null`
como dice el formato acordado.

In [ ]:
def spacy_agreement_problems(doc) -> list[str]:
    """Desacuerdos que se pueden ver con los rasgos de spaCy. Si falta un rasgo, no se puede comparar."""
    problems = []
    for t in doc:
        if t.dep_ in ("det", "amod"):
            for feature in ("Gender", "Number"):
                a, b = t.morph.get(feature), t.head.morph.get(feature)
                if a and b and a != b:
                    problems.append(f"{t.text}/{t.head.text}: {feature} {a[0]}≠{b[0]}")
        if t.dep_ == "nsubj" and t.head.dep_ == "ROOT" and t.head.pos_ in ("VERB", "AUX"):
            a, b = t.morph.get("Number"), t.head.morph.get("Number")
            if a and b and a != b:
                problems.append(f"{t.text}/{t.head.text}: sujeto-verbo {a[0]}≠{b[0]}")
    return problems


rows = []
for item in items:
    problems = spacy_agreement_problems(docs[item.id])
    rows.append({
        "id": item.id, "oracion": item.sentence,
        "error_esperado": item.error_type or "ninguno",
        "spacy_detecta": "; ".join(problems) or "nada",
        "coincide": (item.error_type == "concordancia") == bool(problems),
    })
pd.DataFrame(rows)

### 1.5 Unión de las frases preposicionales (`pp_adjunta`)

El conjunto de prueba marca a qué se une la **primera** PP de cada oración según la lectura de un hablante
(`sustantivo`, `verbo` o `ambigua`; sección 2.3 del documento). Esto conecta la comparación con el conflicto
LL(1) de nuestra gramática (sección 1.4):

- **Nuestro parser** aplica una regla fija: si el verbo tiene objeto, la PP va al sustantivo más cercano.
  Nunca responde `ambigua`.
- **spaCy** decide con su modelo estadístico, a partir de la dependencia: `obl` (unida al verbo) o `nmod`
  (unida al sustantivo). Tampoco puede responder `ambigua`: siempre da un solo árbol.
- **El LLM** puede responder `ambigua` (sección 3, cuando haya resultados).

El resultado del parser es `results/parser.jsonl` (`python -m evaluation.run_parser`). Si el parser
rechaza la oración no hay árbol, y su `pp_adjunta` cuenta como "sin respuesta".

In [ ]:
from experiments import reports

parser_pp = load_predictions("parser") if results_path("parser").exists() else None
print("Parser:", "results/parser.jsonl" if parser_pp else "sin resultados (correr python -m evaluation.run_parser)")

pp_rows = []
for item, pred in zip(items, spacy_predictions):
    if item.pp_attachment is None:
        continue
    doc = docs[item.id]
    prep = next(t for t in doc if t.pos_ == "ADP" and t.dep_ == "case")
    pp_rows.append({
        "id": item.id, "oracion": item.sentence, "esperado": item.pp_attachment,
        "spacy": pred.pp_attachment,
        "spacy_dependencia": f"{prep.text} → {prep.head.text} ({prep.head.dep_}) → {prep.head.head.text}",
        "parser": next((p.pp_attachment for p in parser_pp if p.id == item.id), None) if parser_pp else None,
    })
pp_table = pd.DataFrame(pp_rows)
pp_table.to_csv(REPORTS / "pp_adjunta_comparacion.csv", index=False)
pp_table

In [ ]:
for name, preds in (("spacy", spacy_predictions), ("parser", parser_pp)):
    if preds is None:
        continue
    matrix, summary = reports.pp_attachment_report(items, preds)
    print(f"\n=== {name}")
    display(matrix)
    display(summary.to_frame("valor").T)

### 1.6 Sondeo de robustez

Variaciones mínimas de algunas oraciones, para separar lo que spaCy "sabe" de los accidentes del modelo:
la mayúscula inicial, oraciones muy cortas y pares mínimos correcto/incorrecto. No son parte del conjunto
de prueba.

In [ ]:
PROBES = [
    ("mayúscula inicial", "Come la manzana."),
    ("mayúscula inicial", "come la manzana."),
    ("sujeto explícito", "Ella come la manzana."),
    ("oración corta, incorrecta", "Los perros come."),
    ("oración corta, correcta", "Los perros comen."),
    ("oración corta, correcta", "El perro come."),
    ("concordancia, incorrecta", "La elefante grises duerme."),
    ("concordancia, correcta", "El elefante gris duerme."),
]

pd.DataFrame([{
    "caso": case, "oracion": sentence,
    "analisis": " ".join(f"{t.text}/{t.pos_}/{t.dep_}" for t in nlp(sentence) if t.pos_ != "PUNCT"),
} for case, sentence in PROBES])

### 1.7 Matriz de confusión y reporte de clasificación

Métricas de **etiquetado** palabra por palabra (sección 2.7 b) y de **funciones sintácticas** (2.7 c).
spaCy no decide si la oración es válida (`acepta` = `null`), así que no tiene matriz de aceptación: esa
está en las secciones 2.5 y 3 para los sistemas que sí deciden (parser, Transformers y LLM).

Las oraciones con un número distinto de palabras no se pueden comparar palabra por palabra; se
excluyen y se reportan aparte. Las figuras y tablas se guardan en `results/figures/` y `results/reports/`
para el informe.

In [ ]:
import matplotlib.pyplot as plt

from experiments import reports

y_true, y_pred, skipped = reports.tag_pairs(items, spacy_predictions)
print(f"Palabras comparadas: {len(y_true)} | oraciones excluidas por distinto número de palabras: {skipped or 'ninguna'}")

spacy_confusion = reports.tag_confusion(y_true, y_pred)
spacy_confusion

In [ ]:
reports.plot_confusion(spacy_confusion, "spaCy: categorías esperadas vs. predichas",
                       FIGURES / "spacy_matriz_confusion.png")
plt.show()

In [ ]:
spacy_report = reports.tag_report(y_true, y_pred)
spacy_report.to_csv(REPORTS / "spacy_reporte_clasificacion.csv")
spacy_report.round(3)

In [ ]:
spacy_functions = reports.function_report(items, spacy_predictions)
spacy_functions.to_csv(REPORTS / "spacy_funciones.csv")
spacy_functions

In [ ]:
# Resumen con todas las métricas de spaCy, también por nivel.
def spacy_summary(subset, name):
    ids = {i.id for i in subset}
    preds = [p for p in spacy_predictions if p.id in ids]
    yt, yp, skip = reports.tag_pairs(subset, preds)
    rep = reports.tag_report(yt, yp) if yt else None
    fun = reports.function_report(subset, preds)
    times = [p.time_ms for p in preds if not p.run_error]
    return {
        "nivel": name,
        "oraciones": len(subset),
        "palabras": len(yt),
        "exactitud_etiquetas": rep.loc["exactitud", "f1"] if rep is not None else None,
        "precision_macro": rep.loc["promedio macro", "precision"] if rep is not None else None,
        "cobertura_macro": rep.loc["promedio macro", "cobertura"] if rep is not None else None,
        "f1_macro": rep.loc["promedio macro", "f1"] if rep is not None else None,
        "f1_ponderado": rep.loc["promedio ponderado", "f1"] if rep is not None else None,
        "excluidas": len(skip),
        "sujeto": fun.loc["sujeto", "exactitud"],
        "verbo": fun.loc["verbo", "exactitud"],
        "objeto": fun.loc["objeto", "exactitud"],
        "tiempo_ms_promedio": sum(times) / len(times) if times else None,
    }


spacy_all = pd.DataFrame(
    [spacy_summary(items, "todos")]
    + [spacy_summary([i for i in items if i.level == lv], str(lv)) for lv in (1, 2, 3)]
).set_index("nivel")
spacy_all.to_csv(REPORTS / "spacy_resumen.csv")
spacy_all.round(3)

### 1.8 Hallazgos de spaCy

> Con las 73 oraciones del conjunto de prueba y spaCy 3.8 con `es_core_news_sm` 3.8.0.

1. **Etiqueta bien casi todo, pero falla en los verbos.** Acierta 359 de 381 palabras (exactitud 0,942;
   F1 macro 0,898). De los 22 errores, 14 son verbos etiquetados como otra cosa: 10 como sustantivo, 3 como
   adjetivo y 1 como determinante. Por eso `Verb` tiene cobertura 0,816 y `Noun` precisión 0,901. En las
   45 oraciones de la gramática da todas las etiquetas bien en 38, y sujeto, verbo y objeto bien en 40.
2. **Lo más frágil son las oraciones cortas, no las difíciles.** *"El perro come."*, *"El profesor lee."*,
   *"Tú compras las manzanas."* y *"El perro de los niños come."* salen con el verbo como sustantivo o
   adjetivo. Por eso el nivel 1 tiene la exactitud de etiquetado más baja (0,924, frente a 0,959 del nivel 2).
   Con sujeto y verbo solos, al modelo pequeño le falta contexto.
3. **Con sus rasgos detecta parte de la concordancia, pero no es confiable.** La heurística de la sección 1.4
   encuentra algo en 8 de las 13 oraciones con error de concordancia. Se le escapan 5. En *"La elefante
   grises duerme."* toma *elefante* como nombre propio, sin rasgos; en *"Los perros come."* y *"La perro
   come."* el verbo sale como sustantivo. Además da 2 falsas alarmas en oraciones correctas: *"Los gatos
   duermen."* y *"Tus perras comen."*. La CFG con rasgos rechaza las 13 sin falsas alarmas.
4. **Por la mayúscula inicial, no analiza bien el sujeto omitido.** *"Come la manzana."* sale con *Come*
   como nombre propio, pero *"come la manzana."* sale bien (sección 1.6). Es justo un caso que la CFG **no
   puede** generar y spaCy **sí** podría, si no fuera por la mayúscula.
5. **Unión de las PP (sección 1.5): spaCy acierta 13 de 17; el parser, 10 de 17.** spaCy falla en dos
   oraciones, *"…los árboles altos desde la casa"* y *"…unas manzanas muy rojas para sus niños"*, donde
   une la PP al sustantivo. El parser falla en esas dos y en otras dos con objeto, *"…la manzana con su
   perro"* y *"…su pelota azul en el parque"*, porque su regla fija pega la PP al sustantivo más cercano.
   **Ninguno detecta las 2 ambiguas:** spaCy elige el verbo y el parser el sustantivo. Es el conflicto
   LL(1) de la sección 1.4, medido.
6. **Palabras fuera del léxico:** el conjunto de prueba les pone `X` (*que*, *no*). spaCy les da su
   categoría real (`Prn`, `Adv`). Cuenta como error, pero es una diferencia de convención, no un fallo.
7. **Tiempo:** mediana de 1,6 ms por oración. La primera tarda unos 8 ms porque incluye la preparación del
   modelo.
8. **Errores que no cambian las métricas:** *come* con modo subjuntivo (`Mood=Sub`) en *"Ella come una
   manzana."*, *ayer* unido a *gris* en vez de al verbo, y *pijamas* marcado como femenino.

TODO(P1): probar un modelo más grande (`es_core_news_md`, `es_core_news_lg` o el de transformers,
`es_dep_news_trf`) para ver si los hallazgos 2 y 4 son límites del modelo pequeño o de spaCy en general.

## 2. HuggingFace Transformers

Se prueban **dos** modelos de clasificación de tokens para español que dan etiquetas **UPOS** (así se
usa la misma conversión de la sección 2.5):

| Clave | Modelo | Base | Parámetros | Pesos | Documentación |
| --- | --- | --- | --- | --- | --- |
| `xlmr_ud28` | `wietsedv/xlm-roberta-base-ft-udpos28-es` | XLM-RoBERTa base (multilingüe) | 277 M | 1,1 GB | Universal Dependencies 2.8; exactitud 95,5 % en el test de español (ficha del modelo); Apache-2.0 |
| `beto16` | `mrm8488/bert-spanish-cased-finetuned-pos-16-tags` | BETO (BERT en español) | 109 M | 440 MB | Sin ficha: no se publican los datos de entrenamiento ni la exactitud |

**Oficial:** el formato acordado tiene un solo `results/transformers.jsonl`. Se usa `xlmr_ud28` porque es el
que se puede documentar en el informe; `beto16` se guarda en `results/reports/` y se compara aquí. Para
cambiarlo, editar `OFFICIAL` abajo.

**Cómo se etiqueta:** se le pasan al modelo las palabras ya separadas y cada palabra toma la etiqueta de
su primer fragmento (`experiments/hf_tagger.py`). Así hay exactamente una etiqueta por palabra; el
`pipeline` de HuggingFace con agrupación puede juntar palabras vecinas con la misma etiqueta.

**Recursos (medidos en una Mac con 8 GB):** se carga un modelo a la vez; el proceso llega a ~1 GB de RAM.
La primera vez cada modelo se descarga a `~/.cache/huggingface`. Instalar antes `torch` y `transformers`
(`requirements-experiments.txt`).

In [ ]:
import importlib.util

MODELS = {
    # clave: (nombre en el Hub, use_safetensors)
    "xlmr_ud28": ("wietsedv/xlm-roberta-base-ft-udpos28-es", False),   # solo publica pytorch_model.bin
    "beto16": ("mrm8488/bert-spanish-cased-finetuned-pos-16-tags", None),
}
OFFICIAL = "xlmr_ud28"

TRANSFORMERS_READY = all(importlib.util.find_spec(m) for m in ("torch", "transformers"))
if not TRANSFORMERS_READY:
    print("Falta instalar torch y transformers (requirements-experiments.txt); se salta esta sección.")

In [ ]:
from evaluation.formats import write_jsonl
from experiments.hf_tagger import HFTagger
from experiments.tagsets import upos_category

# Variantes para el sondeo de robustez (mayúscula inicial, oraciones cortas).
HF_PROBES = [
    "Come la manzana", "come la manzana", "Él mató un elefante", "él mató un elefante",
    "Los perros come", "Los perros comen", "El perro come", "La elefante grises duerme",
]

hf_predictions, hf_info, hf_probe_rows = {}, {}, []
if TRANSFORMERS_READY:
    for key, (model_name, use_safetensors) in MODELS.items():
        tagger = HFTagger(model_name, use_safetensors=use_safetensors)   # un modelo a la vez
        hf_info[key] = {"modelo": model_name, "parametros_M": round(tagger.parameters / 1e6),
                        "carga_s": round(tagger.load_seconds, 1)}
        predictions = []
        for item in items:
            words = item.sentence.rstrip().rstrip(".").split()
            start = time.perf_counter()
            try:
                raw = tagger.tag(words)
                tags = [[w.lower(), upos_category(t, w) or "X"] for w, t in zip(words, raw)]
                predictions.append(Prediction(item.id, "transformers", accepts=None, tags=tags,
                                              time_ms=round((time.perf_counter() - start) * 1000, 3),
                                              ))
            except Exception as e:  # el formato pide escribir la línea aunque falle
                predictions.append(Prediction(item.id, "transformers", accepts=None,
                                              run_error=f"{type(e).__name__}: {e}"))
        hf_predictions[key] = predictions
        for probe in HF_PROBES:
            words = probe.split()
            hf_probe_rows.append({"modelo": key, "oracion": probe,
                                  "etiquetas": " ".join(f"{w}/{t}" for w, t in zip(words, tagger.tag(words)))})
        tagger.close()
        del tagger

    print("Oficial:", save_predictions("transformers", hf_predictions[OFFICIAL], items))
    for key, predictions in hf_predictions.items():
        if key != OFFICIAL:
            path = REPORTS / f"transformers_{key}.jsonl"
            write_jsonl(path, (p.to_json() for p in predictions))
            print("Comparación:", path)
    display(pd.DataFrame(hf_info).T)

### 2.1 Matriz de confusión y reporte de clasificación de cada modelo

In [ ]:
hf_reports = {}
for key, predictions in hf_predictions.items():
    yt, yp, skip = reports.tag_pairs(items, predictions)
    print(f"\n=== {key} ({MODELS[key][0]}) | palabras {len(yt)} | excluidas {skip or 'ninguna'}")
    matrix = reports.tag_confusion(yt, yp)
    display(matrix)
    reports.plot_confusion(matrix, f"{key}: categorías esperadas vs. predichas",
                           FIGURES / f"transformers_{key}_matriz_confusion.png")
    plt.show()
    hf_reports[key] = reports.tag_report(yt, yp)
    hf_reports[key].to_csv(REPORTS / f"transformers_{key}_reporte_clasificacion.csv")
    display(hf_reports[key].round(3))

### 2.2 Comparación de los tres etiquetadores

Etiquetado palabra por palabra contra el conjunto de prueba. Los dos modelos de Transformers solo
etiquetan (no dan sujeto, verbo ni objeto).

In [ ]:
def tagger_summary(name, predictions):
    yt, yp, skip = reports.tag_pairs(items, predictions)
    rep = reports.tag_report(yt, yp)
    times = [p.time_ms for p in predictions if not p.run_error]
    wrong = [f"{w}: {e}→{g}" for i in items for p in predictions if p.id == i.id and p.tags
             and len(p.tags) == len(i.tags) for (w, e), (_, g) in zip(i.tags, p.tags) if e != g]
    return {"sistema": name, "exactitud": rep.loc["exactitud", "f1"],
            "precision_macro": rep.loc["promedio macro", "precision"],
            "cobertura_macro": rep.loc["promedio macro", "cobertura"],
            "f1_macro": rep.loc["promedio macro", "f1"], "excluidas": len(skip),
            # Mediana: la primera oración incluye la preparación del modelo (~2-3 s) y distorsiona el promedio.
            "ms_por_oracion_mediana": float(pd.Series(times).median()) if times else None,
            "errores": "; ".join(wrong) or "ninguno"}


comparison = pd.DataFrame(
    [tagger_summary("spacy", spacy_predictions)]
    + [tagger_summary(key, preds) for key, preds in hf_predictions.items()]
).set_index("sistema")
comparison.to_csv(REPORTS / "etiquetadores_comparacion.csv")
comparison.round(3)

In [ ]:
# Oración por oración: ¿quién acierta todas las etiquetas?
def all_tags_ok(item, pred):
    return pred.tags is not None and [c for _, c in pred.tags] == [c for _, c in item.tags]

taggers = {"spacy": spacy_predictions, **hf_predictions}
pd.DataFrame([
    {"id": i.id, "oracion": i.sentence, "en_gramatica": i.in_grammar,
     **{name: "✓" if all_tags_ok(i, next(p for p in preds if p.id == i.id)) else "✗"
        for name, preds in taggers.items()}}
    for i in items
])

### 2.3 Sondeo de robustez (Transformers)

Las mismas variantes que con spaCy (sección 1.6), con las etiquetas UPOS crudas de cada modelo.

In [ ]:
pd.DataFrame(hf_probe_rows).pivot(index="oracion", columns="modelo", values="etiquetas").loc[HF_PROBES]

### 2.4 Hallazgos de Transformers

> Con las 73 oraciones del conjunto de prueba, torch 2.15 y transformers 5.24.

1. **Los dos modelos etiquetan mejor que spaCy.**

   | Etiquetador | Palabras bien (de 381) | Exactitud | F1 macro |
   | --- | --- | --- | --- |
   | BETO-POS (`beto16`) | 376 | 0,987 | 0,970 |
   | XLM-R (`xlmr_ud28`) | 375 | 0,984 | 0,952 |
   | spaCy | 359 | 0,942 | 0,898 |

   Los pocos errores de cada modelo son de tipos distintos:
   - **XLM-R:** *Come* como preposición en *"Come la manzana."*, *perro* y *come* intercambiados en
     *"Perro el come."* (un orden imposible), y tres palabras `X` del conjunto (*que*, *no*) con su
     categoría real.
   - **BETO-POS:** *Él* como nombre propio (oraciones 3, 4 y 26), *yo* como adverbio y *no* como
     adverbio.
2. **Aciertan donde falla spaCy en oraciones cortas.** Las 7 oraciones de la gramática en las que spaCy
   se equivoca (*"El perro come."*, *"El profesor lee."*…) salen bien con XLM-R. XLM-R acierta las 45
   oraciones de la gramática.
3. **La mayúscula inicial no afecta igual a todos** (sección 2.3):
   - BETO-POS etiqueta *Él* como `PROPN`, pero *él* (minúscula) como `PRON`: es la mayúscula.
   - XLM-R etiqueta *come* como `ADP` **también en minúscula**: su problema es el verbo al inicio sin
     sujeto, no la mayúscula.
   - BETO-POS es el único de los tres que acierta en *"Come la manzana"*, el caso de sujeto omitido.
4. **Ninguno de los dos etiquetadores detecta la concordancia.** XLM-R etiqueta bien las 13 oraciones con
   error de concordancia, como si fueran válidas. Rechazarlas es justamente lo que aporta la CFG con
   rasgos.
5. **Costo:** la primera oración tarda ~2–3 s, porque incluye la preparación del modelo. Después, la
   mediana es de ~24 ms por oración en CPU, un orden de magnitud más que spaCy (tabla 2.2). Cargar el
   modelo tarda ~1–3 minutos la primera vez, por la descarga, y unos segundos después. Espacio: 1,1 GB
   (XLM-R) y 440 MB (BETO-POS).

### 2.5 Gramaticalidad con BETO: pseudo-log-verosimilitud

Un etiquetador no rechaza oraciones. Para que Transformers tenga un valor real de `acepta` se usa un
**modelo de lenguaje con máscara** en español, BETO (`dccuchile/bert-base-spanish-wwm-cased`, 110 M de
parámetros, 440 MB), sin entrenar nada (`experiments/acceptability.py`):

1. **Puntuación:** pseudo-log-verosimilitud (PLL; Salazar et al., 2020). Se enmascara cada fragmento de
   la oración y se mide qué probabilidad le da el modelo al fragmento original. Se usa la variante
   *word-l2r* de Kauf e Ivanova (2023), que enmascara también los fragmentos siguientes de la misma
   palabra, y se promedia por fragmento para no castigar las oraciones largas. Mayor = más aceptable.
2. **Umbral:** se calibra con `data/calibration_acceptability.csv`, 12 pares mínimos correcto/incorrecto
   que **no** están en el conjunto de prueba. Se elige el umbral con mayor exactitud en esas oraciones.
3. **Decisión:** `acepta = puntuación >= umbral`. `tipo_error` queda en `null`: el modelo dice qué tan
   natural es la oración, no *por qué* está mal.

Lo que mide BETO es "¿suena a español correcto?", así que su referencia natural es `espanol_valido`.
Contra `en_gramatica` se ve si coincide con nuestras reglas.

In [ ]:
import csv

from experiments.acceptability import PLLScorer, calibrate_threshold

calibration = list(csv.DictReader((ROOT / "data" / "calibration_acceptability.csv").open(encoding="utf-8")))
pll_rows, cal_scores, test_scores = [], [], {}
if TRANSFORMERS_READY:
    scorer = PLLScorer()
    cal_scores = [scorer.score(r["oracion"]) for r in calibration]
    THRESHOLD, cal_accuracy = calibrate_threshold(cal_scores, [r["espanol_valido"] == "si" for r in calibration])
    for item in items:
        start = time.perf_counter()
        test_scores[item.id] = (scorer.score(item.sentence), (time.perf_counter() - start) * 1000)
    scorer.close()
    del scorer
    print(f"Umbral calibrado: {THRESHOLD:.3f} | exactitud en calibración: {cal_accuracy:.3f}")

In [ ]:
# Pares mínimos de calibración: ¿la versión correcta puntúa más alto que la incorrecta?
pairs = {}
for row, score in zip(calibration, cal_scores):
    pairs.setdefault(row["par"], {})[row["espanol_valido"]] = (row["oracion"], score, row["tipo"])
pair_table = pd.DataFrame([
    {"par": k, "tipo": v["si"][2], "correcta": v["si"][0], "punt_correcta": v["si"][1],
     "incorrecta": v["no"][0], "punt_incorrecta": v["no"][1], "gana_la_correcta": v["si"][1] > v["no"][1]}
    for k, v in pairs.items()
])
print(f"La correcta puntúa más alto en {pair_table.gana_la_correcta.sum()}/{len(pair_table)} pares")
pair_table.round(3)

In [ ]:
# Puntuaciones del conjunto de prueba frente al umbral.
pll_table = pd.DataFrame([
    {"id": i.id, "oracion": i.sentence, "puntuacion": test_scores[i.id][0],
     "acepta": test_scores[i.id][0] >= THRESHOLD,
     "espanol_valido": i.valid_spanish, "en_gramatica": i.in_grammar}
    for i in items
]) if test_scores else pd.DataFrame()
if not pll_table.empty:
    pll_table.to_csv(REPORTS / "transformers_pll.csv", index=False)

    fig, ax = plt.subplots(figsize=(8, 3.5))
    cal_good = [s for r, s in zip(calibration, cal_scores) if r["espanol_valido"] == "si"]
    cal_bad = [s for r, s in zip(calibration, cal_scores) if r["espanol_valido"] == "no"]
    ax.scatter(cal_good, [2] * len(cal_good), marker="o", label="calibración: correcta")
    ax.scatter(cal_bad, [2] * len(cal_bad), marker="x", label="calibración: incorrecta")
    good = pll_table[pll_table.espanol_valido]
    bad = pll_table[~pll_table.espanol_valido]
    ax.scatter(good.puntuacion, [1] * len(good), marker="o", label="prueba: español válido")
    ax.scatter(bad.puntuacion, [1] * len(bad), marker="x", label="prueba: inválido")
    for _, r in pll_table.iterrows():
        ax.annotate(str(r["id"]), (r["puntuacion"], 1), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=8)
    ax.axvline(THRESHOLD, linestyle="--", color="gray", label=f"umbral {THRESHOLD:.2f}")
    ax.set_yticks([1, 2], ["prueba", "calibración"])
    ax.set_xlabel("PLL promedio por fragmento (mayor = más aceptable)")
    ax.set_ylim(0.5, 2.5)
    ax.legend(fontsize=7, loc="lower left")
    fig.tight_layout()
    fig.savefig(FIGURES / "transformers_pll_umbral.png", dpi=150)
    plt.show()
pll_table.round(3)

In [ ]:
# acepta y tiempo de BETO en el resultado oficial de Transformers (las etiquetas siguen siendo de XLM-R).
if test_scores:
    for pred in hf_predictions[OFFICIAL]:
        score, ms = test_scores[pred.id]
        pred.accepts = score >= THRESHOLD
        pred.error_type = None        # un modelo de lenguaje no sabe por qué una oración está mal
        pred.time_ms = round(pred.time_ms + ms, 3)
    print("Oficial con acepta:", save_predictions("transformers", hf_predictions[OFFICIAL], items))

    for reference in ("valid_spanish", "in_grammar"):
        matrix = reports.acceptance_confusion(items, hf_predictions[OFFICIAL], reference)
        print(f"\nTransformers (BETO) contra {reports.REFERENCES[reference]}")
        display(matrix)
        display(reports.acceptance_report(matrix).to_frame("valor").T)

### 2.6 Hallazgos de la gramaticalidad con BETO

> Con las 73 oraciones del conjunto de prueba y las 24 de calibración.

1. **En pares mínimos, BETO distingue siempre.** En los 12 pares de calibración la versión correcta
   puntúa más alto que la incorrecta, tanto en concordancia (*"El perro duerme"* / *"El perro duermen"*)
   como en orden de palabras.
2. **Con un umbral absoluto, falla bastante más.** Con el umbral calibrado (−3,881):
   - **Contra `espanol_valido`:** precisión 0,979, cobertura 0,821 (46 de 56), rechazo correcto 0,941 y
     exactitud 0,849.
   - **Acepta solo una oración incorrecta:** *"La niña comen el pan."*.
   - **Rechaza 10 correctas:** 9 de la gramática (2, 3, 4, 12, 19, 24, 26, 27, 30) y *"Compraron ellos
     los libros ayer."*.
3. **Por qué:** la PLL mezcla gramaticalidad con **frecuencia**. Las oraciones rechazadas tienen palabras
   poco frecuentes (*durmieron*, *elefante*, *gris*, *pijamas*, *perras*) o son muy cortas (*"Mi perro
   duerme."*, *"Nosotras dormimos."*). En ambos casos, cada fragmento tiene poco contexto y la
   puntuación baja aunque la oración sea correcta. Comparar dos versiones de la misma oración cancela ese
   efecto; un umbral absoluto no. Una mejora conocida es normalizar por la frecuencia de cada palabra
   (SLOR; Lau et al., 2017).
4. **Sigue al español, no a nuestras reglas.** Acepta 10 de las 11 oraciones válidas que la gramática no
   genera (sujeto omitido, coordinación, subordinadas…). Contra `en_gramatica` eso cuenta como falsos
   positivos: exactitud 0,726 y rechazo correcto 0,607. Es la misma pregunta que el experimento del LLM:
   ¿reglas o conocimiento del español?
5. **Costo:** BETO pesa 440 MB. Hace una pasada por cada fragmento de la oración, así que Transformers
   completo (etiquetar con XLM-R y puntuar con BETO) tiene una mediana de 74 ms por oración en CPU.

## 3. Comparación con la CFG

Se comparan los cuatro análisis del mismo conjunto de prueba (73 oraciones):

| Sistema | Archivo | Decide (`acepta`) | Etiquetas | Funciones | `pp_adjunta` |
| --- | --- | --- | --- | --- | --- |
| **Parser (CFG)** | `results/parser.jsonl` | ✔ con `tipo_error` | solo si hay árbol | solo si hay árbol | solo si hay árbol |
| **spaCy** | `results/spacy.jsonl` | — | ✔ | ✔ | ✔ |
| **Transformers** (XLM-R etiqueta, BETO juzga) | `results/transformers.jsonl` | ✔ sin `tipo_error` | ✔ | — | — |
| BETO-POS (segundo etiquetador) | `results/reports/transformers_beto16.jsonl` | — | ✔ | — | — |

El parser solo da etiquetas, funciones y PP cuando construye un árbol: en las oraciones aceptadas y en las
rechazadas **solo** por concordancia. En las demás no da nada. Por eso, además de la tabla general, el
etiquetado y las funciones se comparan **en el mismo subconjunto** (sección 3.4): si no, el parser se
compararía solo en las oraciones que ya sabe analizar.

Las métricas de la tabla general salen de `evaluation/metrics.py` (P4), así que coinciden con
`results/metricas.md`.

In [ ]:
from evaluation import metrics
from evaluation.formats import read_jsonl
from evaluation.plots import SYSTEM_COLORS

NAMES = {"parser": "Parser (CFG)", "spacy": "spaCy", "transformers": "Transformers", "beto16": "BETO-POS"}
preds = {s: {p.id: p for p in load_predictions(s)} for s in ("parser", "spacy", "transformers")
         if results_path(s).exists()}
BETO16_PATH = REPORTS / "transformers_beto16.jsonl"
if BETO16_PATH.exists():
    preds["beto16"] = {d["id"]: Prediction.from_json(d) for d in read_jsonl(BETO16_PATH)}
missing = [s for s in ("parser", "spacy", "transformers") if s not in preds]
if missing:
    print("Faltan resultados de:", ", ".join(missing), "(correr las secciones 1 y 2 y python -m evaluation.run_parser)")


def tags_ok(item, pred) -> bool:
    return pred is not None and pred.tags is not None and [c for _, c in pred.tags] == [c for _, c in item.tags]


def functions_ok(item, pred) -> bool:
    return pred is not None and pred.subject is not None and \
        (pred.subject, pred.verb, pred.object) == (item.subject, item.verb, item.object)


metric_rows = metrics.compute_all(metrics.load_all_pairs(items))
val = lambda s, comp, m, level="todos": metrics.value(metric_rows, s, comp, m, level)
summary_rows = {
    "aceptación: exactitud contra en_gramatica": lambda s: val(s, "aceptacion", "exactitud"),
    "aceptación: exactitud contra espanol_valido": lambda s: val(s, "aceptacion_espanol", "exactitud"),
    "tipo_error correcto (rechazos)": lambda s: val(s, "aceptacion", "tipo_error"),
    "oraciones con etiquetas": lambda s: sum(p.tags is not None for p in preds[s].values()),
    "etiquetado: exactitud (en las que etiqueta)": lambda s: val(s, "etiquetado", "exactitud"),
    "sujeto": lambda s: val(s, "funciones", "sujeto"),
    "verbo": lambda s: val(s, "funciones", "verbo"),
    "objeto": lambda s: val(s, "funciones", "objeto"),
    "pp_adjunta: exactitud": lambda s: val(s, "estructura", "pp_adjunta"),
    "ms por oración (mediana)": lambda s: pd.Series([p.time_ms for p in preds[s].values()]).median(),
}
main_systems = [s for s in ("parser", "spacy", "transformers") if s in preds]
comparison_summary = pd.DataFrame(
    {NAMES[s]: {name: f(s) for name, f in summary_rows.items()} for s in main_systems}
)
comparison_summary.to_csv(REPORTS / "comparacion_cfg_resumen.csv")
comparison_summary.round(3).astype(object).fillna("—")

### 3.1 Por tipo de oración: las dos preguntas del notebook

El conjunto de prueba tiene cuatro grupos según `en_gramatica`, `espanol_valido` y `tipo_error`. Para cada
uno se cuenta:

- **Decisión** (parser y Transformers): cuántas oraciones acepta cada uno. Lo esperado según las reglas es
  aceptar solo el primer grupo; según el español, los dos primeros.
- **Análisis** (todos): en cuántas da **todas** las etiquetas correctas, y en cuántas sujeto, verbo y objeto
  correctos.

El grupo 2 responde la pregunta 1 (lo que la CFG no puede) y el grupo 3, la pregunta 2 (lo que la CFG
acierta y los demás no).

In [ ]:
GROUPS = {
    "1. en la gramática": lambda i: i.in_grammar,
    "2. fuera de la gramática, español válido": lambda i: not i.in_grammar and i.valid_spanish,
    "3. error de concordancia": lambda i: i.error_type == "concordancia",
    "4. error sintáctico, no es español": lambda i: i.error_type == "sintactico" and not i.valid_spanish,
}
group_of = {i.id: next(g for g, f in GROUPS.items() if f(i)) for i in items}
assert all(sum(f(i) for f in GROUPS.values()) == 1 for i in items), "cada oración va en un solo grupo"

group_rows = []
for group, belongs in GROUPS.items():
    members = [i for i in items if belongs(i)]
    row = {"grupo": group, "oraciones": len(members)}
    for s in ("parser", "transformers"):
        if s in preds:
            row[f"acepta: {NAMES[s]}"] = sum(preds[s][i.id].accepts is True for i in members)
    for s in ("parser", "spacy", "transformers", "beto16"):
        if s in preds:
            row[f"etiquetas ok: {NAMES[s]}"] = sum(tags_ok(i, preds[s][i.id]) for i in members)
    for s in ("parser", "spacy"):
        if s in preds:
            row[f"funciones ok: {NAMES[s]}"] = sum(functions_ok(i, preds[s][i.id]) for i in members)
    group_rows.append(row)
group_table = pd.DataFrame(group_rows).set_index("grupo")
group_table.to_csv(REPORTS / "comparacion_cfg_grupos.csv")
group_table.T

In [ ]:
# Figura para el informe: proporción de oraciones de cada grupo que cada sistema acepta (izquierda) y
# analiza completo, con todas las etiquetas correctas (derecha).
from evaluation import plots

short = dict(zip(GROUPS, ["en la\ngramática", "fuera, pero\nespañol válido",
                           "error de\nconcordancia", "error sintáctico\n(no es español)"]))
sizes = group_table["oraciones"]
accept = {s: [group_table.loc[g, f"acepta: {NAMES[s]}"] / sizes[g] for g in GROUPS]
          for s in ("parser", "transformers") if s in preds}
tagged = {s: [group_table.loc[g, f"etiquetas ok: {NAMES[s]}"] / sizes[g] for g in GROUPS]
          for s in ("parser", "spacy", "transformers") if s in preds}

fig, (left, right) = plots._figure(12, 3.8, ncols=2)
for ax, series, title in ((left, accept, "Oraciones aceptadas"),
                          (right, tagged, "Oraciones con todas las etiquetas correctas")):
    plots._style(ax)
    plots._grouped_bars(ax, [short[g] for g in GROUPS], series, SYSTEM_COLORS, NAMES)
    ax.set_title(title, loc="left", fontsize=10)
    ax.tick_params(axis="x", labelsize=8)
plots._legend(fig, right, 3)
plots._title(fig, "Parser (CFG) frente a spaCy y Transformers, por tipo de oración")
fig.savefig(FIGURES / "comparacion_cfg_grupos.png", dpi=200, bbox_inches="tight", facecolor=plots.SURFACE)

### 3.2 Pregunta 1: lo que spaCy y Transformers analizan y la CFG no puede

Las oraciones con `en_gramatica = no` y `espanol_valido = si`: español correcto que nuestra gramática no
genera (sujeto omitido, orden libre, coordinación, subordinadas, palabras fuera del léxico). El parser las
rechaza todas con error sintáctico, como debe según las reglas, y no da ningún análisis.

In [ ]:
def tag_errors(item, pred) -> str:
    if pred is None or pred.tags is None:
        return "sin análisis"
    wrong = [f"{w}: {e}→{g}" for (w, e), (_, g) in zip(item.tags, pred.tags) if e != g]
    return "; ".join(wrong) or "✓"


q1 = pd.DataFrame([{
    "id": i.id, "oracion": i.sentence, "fenomeno": ", ".join(i.phenomena),
    "parser": "acepta" if preds["parser"][i.id].accepts else f"rechaza ({preds['parser'][i.id].error_type})",
    "spacy_etiquetas": tag_errors(i, preds["spacy"][i.id]),
    "spacy_funciones": "✓" if functions_ok(i, preds["spacy"][i.id]) else
        f"{preds['spacy'][i.id].subject or '∅'} / {preds['spacy'][i.id].verb} / {preds['spacy'][i.id].object or '∅'}",
    "xlmr_etiquetas": tag_errors(i, preds["transformers"][i.id]),
    "beto_acepta": preds["transformers"][i.id].accepts,
} for i in items if group_of[i.id].startswith("2.")])
q1.to_csv(REPORTS / "comparacion_cfg_pregunta1.csv", index=False)
q1

### 3.3 Pregunta 2: lo que la CFG acierta y los demás no

**a) Concordancia.** El parser rechaza las 13 oraciones con error de concordancia y dice que el error es de
concordancia. Los etiquetadores no deciden: sus etiquetas pueden estar bien y la oración seguir siendo
incorrecta. Se muestran además la heurística de rasgos de spaCy (sección 1.4) y la decisión de BETO por
puntuación (sección 2.5).

In [ ]:
q2 = pd.DataFrame([{
    "id": i.id, "oracion": i.sentence,
    "parser": "acepta" if preds["parser"][i.id].accepts else f"rechaza ({preds['parser'][i.id].error_type})",
    "spacy_etiquetas": tag_errors(i, preds["spacy"][i.id]),
    "spacy_rasgos_detectan": "; ".join(spacy_agreement_problems(docs[i.id])) or "nada",
    "xlmr_etiquetas": tag_errors(i, preds["transformers"][i.id]),
    "beto_acepta": preds["transformers"][i.id].accepts,
} for i in items if i.error_type == "concordancia"])
q2.to_csv(REPORTS / "comparacion_cfg_pregunta2.csv", index=False)
print(f"Parser rechaza por concordancia: {(q2['parser'] == 'rechaza (concordancia)').sum()}/{len(q2)}")
print(f"Heurística de rasgos de spaCy detecta algo: {(q2['spacy_rasgos_detectan'] != 'nada').sum()}/{len(q2)}")
print(f"BETO rechaza: {(q2['beto_acepta'] == False).sum()}/{len(q2)}")  # noqa: E712
q2

**b) Análisis de oraciones de la gramática.** Oraciones del grupo 1 en las que el parser da todas las
etiquetas y funciones bien y algún otro sistema falla (o BETO las rechaza).

In [ ]:
q2b = []
for i in items:
    if not i.in_grammar:
        continue
    row = {"id": i.id, "oracion": i.sentence,
           "parser": "✓" if tags_ok(i, preds["parser"][i.id]) and functions_ok(i, preds["parser"][i.id]) else "✗"}
    for s in ("spacy", "transformers", "beto16"):
        if s in preds:
            row[f"{s}_etiquetas"] = tag_errors(i, preds[s][i.id])
    row["spacy_funciones"] = "✓" if functions_ok(i, preds["spacy"][i.id]) else "✗"
    row["beto_acepta"] = preds["transformers"][i.id].accepts
    if any(v not in ("✓", True) for k, v in row.items() if k not in ("id", "oracion")):
        q2b.append(row)
q2b = pd.DataFrame(q2b)
print(f"{len(q2b)} de {sum(i.in_grammar for i in items)} oraciones de la gramática con algún fallo fuera del parser")
q2b

### 3.4 Etiquetado y funciones en el mismo subconjunto

Solo las oraciones en las que el parser construye un árbol (aceptadas o rechazadas por concordancia). Así
los cuatro sistemas se comparan en las mismas palabras.

Ojo al leer el 1,0 del parser: el léxico (`data/lexicon.json`) se escribió para estas oraciones, así que
cada palabra que el parser analiza tiene una sola categoría posible o la gramática la resuelve. Es un
**mundo cerrado**: el parser no generaliza a palabras nuevas (sección 3.2: *corre*, *no*, *que* lo hacen
rechazar), mientras que spaCy y Transformers etiquetan cualquier palabra.

In [ ]:
subset_ids = {i for i, p in preds["parser"].items() if p.tags is not None}
subset = [i for i in items if i.id in subset_ids]
print(f"Subconjunto: {len(subset)} de {len(items)} oraciones, {sum(len(i.tags) for i in subset)} palabras")

f1_columns, subset_rows = {}, []
for s in ("parser", "spacy", "transformers", "beto16"):
    if s not in preds:
        continue
    plist = [preds[s][i.id] for i in subset]
    y_true, y_pred, skipped = reports.tag_pairs(subset, plist)
    report = reports.tag_report(y_true, y_pred)
    f1_columns[NAMES[s]] = report["f1"]
    functions = reports.function_report(subset, plist)["exactitud"] if s in ("parser", "spacy") else None
    subset_rows.append({
        "sistema": NAMES[s],
        "exactitud_etiquetas": report.loc["exactitud", "f1"],
        "f1_macro": report.loc["promedio macro", "f1"],
        "oraciones_todas_bien": sum(tags_ok(i, preds[s][i.id]) for i in subset),
        "sujeto": None if functions is None else functions["sujeto"],
        "verbo": None if functions is None else functions["verbo"],
        "objeto": None if functions is None else functions["objeto"],
    })
subset_table = pd.DataFrame(subset_rows).set_index("sistema")
subset_table.to_csv(REPORTS / "comparacion_cfg_mismo_subconjunto.csv")
display(subset_table.round(3).astype(object).fillna("—"))
print("F1 por categoría en el subconjunto:")
pd.DataFrame(f1_columns).round(3).astype(object).fillna("—")

### 3.5 Aceptación por nivel: reglas frente a español

Parser y Transformers (BETO) son los dos sistemas que deciden. Exactitud por nivel contra las dos
referencias, y las matrices de confusión (sección 2.7 a).

In [ ]:
deciding = [s for s in ("parser", "transformers", "llm_zero", "llm_few")
            if results_path(s).exists() and any(p.accepts is not None for p in load_predictions(s))]
level_table = pd.DataFrame([
    {"sistema": NAMES.get(s, s), "referencia": ref, **{f"nivel {lv}": val(s, comp, "exactitud", lv) for lv in "123"},
     "todos": val(s, comp, "exactitud")}
    for s in deciding
    for ref, comp in (("en_gramatica", "aceptacion"), ("espanol_valido", "aceptacion_espanol"))
]).set_index(["sistema", "referencia"])
level_table.to_csv(REPORTS / "comparacion_cfg_aceptacion_nivel.csv")
display(level_table.round(3))

for system in deciding:
    system_preds = load_predictions(system)
    for reference in ("in_grammar", "valid_spanish"):
        matrix = reports.acceptance_confusion(items, system_preds, reference)
        print(f"\n{NAMES.get(system, system)} contra {reports.REFERENCES[reference]}")
        display(matrix)
        display(reports.acceptance_report(matrix).to_frame("valor").T)

### 3.6 Unión de las frases preposicionales (parser, spaCy y LLM)

`pp_adjunta` contra el conjunto de prueba, en las oraciones que tienen PP (sección 2.7 d). Transformers no
participa: solo etiqueta. El detalle oración por oración está en la sección 1.5.

In [ ]:
structure = [s for s in ("parser", "spacy", "llm_zero", "llm_few") if results_path(s).exists()]
rows = []
for system in structure:
    matrix, summary = reports.pp_attachment_report(items, load_predictions(system))
    print(f"\n=== {NAMES.get(system, system)}")
    display(matrix)
    rows.append({"sistema": NAMES.get(system, system), **summary.to_dict()})
if rows:
    display(pd.DataFrame(rows).set_index("sistema"))

# Dónde no coinciden parser y spaCy, y quién tiene razón.
pp_disagree = pp_table[pp_table["parser"] != pp_table["spacy"]][["id", "oracion", "esperado", "parser", "spacy"]]
pp_disagree

### 3.7 Costo

Mediana de `tiempo_ms` por oración (la media la infla la primera oración, que prepara el modelo) y lo que
ocupa cada sistema en disco. El tiempo de Transformers incluye etiquetar con XLM-R **y** puntuar con BETO.

In [ ]:
import os

lexicon_kb = (ROOT / "data" / "lexicon.json").stat().st_size / 1024
cost = pd.DataFrame([{
    "sistema": NAMES[s],
    "ms_mediana": pd.Series([p.time_ms for p in preds[s].values()]).median(),
    "ms_primera_oracion": preds[s][items[0].id].time_ms,
    "en_disco": {"parser": f"código + léxico de {lexicon_kb:.0f} KB",
                 "spacy": "16 MB (es_core_news_sm)",
                 "transformers": "1,1 GB (XLM-R) + 440 MB (BETO)"}[s],
} for s in main_systems]).set_index("sistema")
cost.to_csv(REPORTS / "comparacion_cfg_costo.csv")
cost.round(2)

### 3.8 Detalle oración por oración

Grupo de la sección 3.1, decisión y si las etiquetas son todas correctas.

In [ ]:
def summary(item, pred):
    if pred is None or pred.run_error:
        return "—"
    decision = "" if pred.accepts is None else ("acepta, " if pred.accepts else
                                                f"rechaza ({pred.error_type or 'sin tipo'}), ")
    return decision + ("etiquetas ✓" if tags_ok(item, pred) else
                       "sin etiquetas" if pred.tags is None else "etiquetas ✗")


pd.set_option("display.max_rows", 100)
pd.DataFrame([
    {"id": i.id, "grupo": group_of[i.id][:2].strip("."), "oracion": i.sentence,
     **{NAMES[s]: summary(i, preds[s].get(i.id)) for s in main_systems}}
    for i in items
]).set_index("id")

### 3.9 Conclusiones (para el informe)

> Con las 73 oraciones. Las cifras salen de las tablas de esta sección y se guardan en
> `results/reports/comparacion_cfg_*.csv`. Falta agregar el LLM cuando existan sus resultados: las secciones
> 3.5 y 3.6 lo incluyen solas.

1. **Cada sistema acierta según la referencia que sigue.**
   - **El parser sigue las reglas al pie de la letra:** exactitud 1,0 contra `en_gramatica`. Acepta las
     45 oraciones de la gramática, rechaza las 28 de fuera y da el `tipo_error` correcto en todas.
   - **Contra `espanol_valido` baja a 0,849:** rechaza las 11 oraciones de español correcto que la
     gramática no genera.
   - **Transformers (BETO) empata en 0,849 contra el español, pero se equivoca en otras oraciones.**
     Rechaza 10 correctas por frecuencia y longitud, y acepta una incorrecta. Contra las reglas cae a
     0,726.
2. **Pregunta 1, lo que ellos analizan y la CFG no puede (11 oraciones).** El parser no da ningún análisis.
   - **Etiquetas correctas:** spaCy en 6, XLM-R en 7 y BETO-POS en 10. Restando las diferencias de
     convención con `X` (*que*, *no*), XLM-R solo falla de verdad en *"Come la manzana."*.
   - **Funciones:** spaCy da sujeto, verbo y objeto correctos en 6.
   - **Lo que resuelven bien:** coordinación (*"El perro y el gato duermen."*), sujeto omitido en pretérito
     (*"Leímos un libro."*) y verbos que no están en el léxico (*"El perro corre en el parque."*).
   - **Lo que les cuesta:** el orden libre y las subordinadas.

   La cobertura es la ventaja de fondo de los modelos estadísticos.
3. **Pregunta 2, lo que la CFG acierta y ellos no.**
   - **Concordancia (13 oraciones).** El parser rechaza las 13 y dice por qué. Los etiquetadores dan buenas
     etiquetas (XLM-R, 13 de 13) pero no las rechazan. La heurística con los rasgos de spaCy detecta 8, con
     2 falsas alarmas en oraciones correctas. BETO rechaza 12, pero sin decir por qué, y también rechaza 9
     oraciones correctas de la gramática, así que su rechazo no señala la concordancia.
   - **Análisis dentro del fragmento.** En las 45 oraciones de la gramática el parser da siempre etiquetas,
     sujeto, verbo y objeto correctos. spaCy falla en 7 (casi todas cortas, con el verbo como sustantivo) y
     BETO-POS en 4.
4. **En el mismo subconjunto (58 oraciones, 304 palabras) el parser y XLM-R empatan en 1,0.** BETO-POS
   queda en 0,987 y spaCy en 0,957. El 1,0 del parser se debe a que trabaja en un **mundo cerrado**: el
   léxico se escribió para estas oraciones. XLM-R llega a lo mismo sin léxico, y además etiqueta las 15
   oraciones restantes.
5. **Unión de las PP: gana spaCy, con 13 de 17 frente a 10 de 17 del parser.** La regla fija del parser
   (pegar la PP al sustantivo más cercano) falla cada vez que la lectura natural une la PP al verbo y hay
   objeto. **Ninguno de los dos detecta las 2 oraciones ambiguas:** los dos dan un solo análisis.
6. **Costo.** Más cobertura cuesta varios órdenes de magnitud de tiempo y espacio:

   | Sistema | Mediana por oración | Espacio |
   | --- | --- | --- |
   | Parser | ~0,01 ms | 14 KB de léxico |
   | spaCy | 1,6 ms | 16 MB |
   | Transformers | 74 ms | 1,5 GB |
7. **Diferencia de fondo.**
   - **La CFG con rasgos es precisa y explicable dentro de su fragmento.** Dice *por qué* rechaza
     (`tipo_error`), pero no tiene cobertura fuera de él: una palabra nueva o una construcción no prevista
     la hacen rechazar.
   - **Los modelos estadísticos tienen cobertura amplia y etiquetan muy bien,** pero no deciden la
     gramaticalidad (spaCy) o la deciden de forma no explicable y sensible a la frecuencia (BETO).
   - **Son complementarios:** un etiquetador estadístico podría reemplazar al léxico cerrado, y la CFG con
     rasgos, verificar la estructura y la concordancia sobre esas etiquetas.